# 01 — Ingest

Fetch raw data from the web and save it to `data/raw/` **untouched**. Nothing in
this notebook modifies data — cleaning/derivation happens in `02-clean`.

## Source: Miller Center presidential speech archive (PRIMARY)

**Publisher:** University of Virginia, Miller Center of Public Affairs.
**What it is:** the Miller Center's curated corpus of U.S. presidential speeches,
George Washington → the contemporary presidency. Offered as a single **bulk
gzipped tar archive** (`https://data.millercenter.org/miller_center_speeches.tgz`)
— the Center deprecated its API in favor of this download. The archive expands to
a `speeches/` directory of JSON files, **one speech per file**, each carrying:
`title`, `doc_name`, `url`, `date`, `transcript` (plain text), `transcript_html`,
`president`, `introduction`.

**License:** public domain (the speeches are U.S. government works); the Miller
Center requests a citation for the compiled archive. No API key / auth required.

**Why this source first:** it's an authoritative university-curated primary corpus
available as one clean file — the right serious-tier starting point for the lead
question (self I/me/my vs collective we/us/our framing by president, and the
1789→present trend). The fuller American Presidency Project (UCSB, ~130k docs) is a
planned later expansion (see `config.yaml`).

**Load-bearing caveat (carry into charts):** inclusion is an **editorial decision**
— this is a curated set of *major* speeches, not every utterance, and coverage is
far denser for modern presidents than 19th-century ones. So "speeches per president"
is a coverage artifact, not behavior. Full methodology + series-break notes are in
`SOURCES.md`.

Config: `sources.miller_center_speeches` (`type: archive_tgz`). Ingest logic:
`src/ingest.py::ingest_miller_center()`.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config, ingest_miller_center
from src.clean_quality import get_connection, load_to_duckdb, register_source, get_sources

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Download + unpack the archive

`ingest_miller_center()` downloads the tgz to `data/raw/miller_center_speeches.tgz`
(cached — re-run is a no-op within 24h), extracts `speeches/*.json` verbatim into
`data/raw/speeches/`, and reads every JSON into one row-per-speech DataFrame. The
raw files are left untouched for provenance.

In [ ]:
df_raw = ingest_miller_center(cfg)
print(df_raw.shape)
df_raw[['president', 'date', 'title']].head(10)

## Quick inspection

Confirm the shape of what landed: columns, date span, speeches-per-president
(the coverage skew the caveat warns about should be visible — modern presidents
have many more speeches than 19th-century ones), and that no transcript is empty.

In [ ]:
print('Columns:', list(df_raw.columns))
print('Date span:', df_raw['date'].str[:10].min(), '→', df_raw['date'].str[:10].max())
print('Presidents:', df_raw['president'].nunique())
print('Empty transcripts:', (df_raw['transcript'].fillna('').str.strip() == '').sum())
print('\nSpeeches per president (top / bottom):')
counts = df_raw['president'].value_counts()
print(counts.head(5).to_string())
print('...')
print(counts.tail(5).to_string())

## Load raw table into DuckDB + register provenance

Load the one-row-per-speech frame into DuckDB as `speeches_raw` (raw ingest table
— cleaning happens in `02-clean`), and record the source in the `_sources`
metadata table so every table's provenance is tracked in the DB.

In [ ]:
from datetime import date

load_to_duckdb(df_raw, 'speeches_raw', con)

register_source(
    con,
    table='speeches_raw',
    name='Miller Center Presidential Speech Archive (UVA)',
    url='https://data.millercenter.org/miller_center_speeches.tgz',
    license='Public domain (U.S. government works); cite the compiled archive.',
    retrieved=date.today().isoformat(),
    notes=(
        'Curated corpus of major U.S. presidential speeches, 1 JSON per speech '
        '(title, date, president, transcript). 1,000+ speeches, Washington to present. '
        'Bulk tgz download; API deprecated. Raw tgz + extracted JSON in data/raw/.'
    ),
    methodology=(
        'Miller Center staff compiled + transcribed a curated set of notable speeches '
        'from public-domain presidential materials. Inclusion is an editorial decision '
        '(not exhaustive); over-represents set-piece addresses (inaugurals, SOTU, major '
        'televised remarks).'
    ),
    series_breaks=(
        'No versioned file breaks. Comparability hazards: (1) coverage density varies by '
        'era — many more speeches for modern presidents than 19th-c., so per-president '
        'aggregates rest on very different sample sizes; (2) written-address era vs '
        'broadcast era — pre-radio messages to Congress were written documents, not '
        'delivered oratory (style break on any 1789→present trend).'
    ),
)

print('speeches_raw rows in DuckDB:',
      con.execute('SELECT COUNT(*) FROM speeches_raw').fetchone()[0])
get_sources(con)

---
**Next:** `02-clean.ipynb` — parse `date` to a real date + `year`, tokenize the
transcript, derive word counts and self/collective pronoun counts, quality-check,
and save interim Parquet.

---
## Cleanup
Close the DuckDB connection so the lock is released for other tools (DBCode, other notebooks). Runs on “Run All”.

In [ ]:
con.close()
print('connection closed')